# Кандидаты для поиска услуг Авито

Решение объединяет текстовый поиск, семантический поиск и историю выбранных объявлений. CatBoost оценивает общий пул и оставляет до 50 кандидатов на запрос.

Для запуска нужны три исходных Parquet-файла в `data/` и `reproduction.zip` из [релиза v1.0.0](https://github.com/NikSila/avito-retrieval-solution/releases/tag/v1.0.0) в корне проекта. Архив содержит обученную модель и эмбеддинги. **Run All** строит индексы и сохраняет `answer.csv`; расчёт выполняется локально.

`RUN_EXPERIMENTS = True` дополнительно запускает обучение и локальную оценку. Поисковые индексы и расчёт признаков реализованы в `src/retrieval.py`.


In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import hashlib
import json
import re
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from threadpoolctl import threadpool_limits
threadpool_limits(4)
from src.retrieval import BM25, CharacterIndex, Retriever, FEATURES, normalize, stem_text, topk

RUN_EXPERIMENTS = False
DATA = Path("data")
EXPECTED_ANSWER = "04a5deafb9f6037ac2c32a069cff1a771f4a22d69b234b5640ff66ca377e0b0b"

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

archive = Path("reproduction.zip")
assert sha256(archive) == "526eaf86065cf8003232e88b4954ec0d1df86957f29c499cfc36959196e0533e"
with zipfile.ZipFile(archive) as bundle:
    bundle.extractall(".")
manifest = json.loads(Path("artifact_manifest.json").read_text())
for name, expected in {**manifest["files"], **manifest["input_files"]}.items():
    assert sha256(name) == expected, f"Другой файл: {name}"
print("Данные и модель проверены.")


Данные и модель проверены.


## 1. Данные и разбиение

В train нет `query_id`, поэтому строки группируются по всем пяти полям `search_*`. Разбиение определяется хешем нормализованного текста: один текст относится к одной части, даже если встречается в разных городах.

70% хеш-корзин используются для истории взаимодействий. Из остальных выбраны 5 000 запросов для обучения отбора, 1 500 для настройки (dev) и 2 000 для окончательной проверки (test).

К `benchmark_items` добавляются выбранные объявления отложенных запросов из train, чтобы все локальные положительные примеры присутствовали в поисковом корпусе. После удаления повторов индекс содержит 200 270 объявлений. При подготовке ответа поиск ограничен исходными 189 212 объявлениями бенчмарка. Порядок строк корпуса должен совпадать с порядком сохранённых эмбеддингов.


In [2]:
SEARCH = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]


def bucket(text):
    return int(hashlib.sha256(text.encode()).hexdigest()[:8], 16) % 100


def read_table(path):
    frame = pd.read_parquet(path)
    for c in ["item_price", "item_latitude", "item_longitude"]:
        if c in frame:
            frame[c] = frame[c].astype(float)
    return frame

train = read_table("data/train.parquet")
bench = read_table("data/benchmark_items.parquet")
queries = read_table("data/benchmark_queries.parquet")
train["normalized_query"] = train.search_query.map(normalize)
train["text_bucket"] = train.normalized_query.map(bucket)
grouped = (
    train.groupby(SEARCH, sort=True, dropna=False)
    .agg(
        relevant=("item_id", lambda values: sorted(set(values))),
        text_bucket=("text_bucket", "first"),
        normalized_query=("normalized_query", "first"),
    )
    .reset_index()
)
# Один текст запроса относится к одной части разбиения независимо от города.
chosen = []
for name, lo, hi, count in [
    ("fit", 70, 85, 5000),
    ("dev", 85, 92, 1500),
    ("test", 92, 100, 2000),
]:
    subset = (
        grouped[grouped.text_bucket.between(lo, hi - 1)]
        .sample(n=count, random_state=42)
        .copy()
    )
    subset["split"] = name
    subset["query_id"] = [
        hashlib.sha256(
            json.dumps([str(x) for x in row], ensure_ascii=False).encode()
        ).hexdigest()[:16]
        for row in subset[SEARCH].itertuples(index=False, name=None)
    ]
    chosen.append(subset)
evaluation = pd.concat(chosen, ignore_index=True)
needed = {x for ids in evaluation.relevant for x in ids}
extra = train[train.item_id.isin(needed)][bench.columns].drop_duplicates("item_id")
# Добавляем к benchmark_items выбранные объявления отложенных запросов,
# чтобы все положительные примеры локальной оценки присутствовали в корпусе.
corpus = (
    pd.concat([bench, extra], ignore_index=True)
    .drop_duplicates("item_id")
    .sort_values("item_id")
    .reset_index(drop=True)
)

print({"train": len(train), "queries": len(queries), "benchmark_items": len(bench), "index_items": len(corpus)})


{'train': 497673, 'queries': 2452, 'benchmark_items': 189212, 'index_items': 200270}


## 2. Поисковые признаки

- BM25 по заголовку, первым 4 000 символам описания и параметрам услуги. Регистр и `ё` нормализуются, русские слова приводятся к основам Snowball.
- Символьный TF-IDF по заголовку — для словоформ и небольших опечаток.
- E5 — для близких по смыслу формулировок. Ближайшие соседи находятся точным поиском по эмбеддингам.
- История похожих запросов даёт вероятность подкатегории и оценки ранее выбранных объявлений.
- География: совпадение локации, расстояние и частоты переходов из локации поиска в города объявлений. Жёсткого ограничения по городу нет.
- Для отбора также используются рейтинг, число отзывов, цена, доступность связи, флаг доставки и совпадение текстовых фильтров. `search_category` почти постоянна и отдельным признаком не используется; `item_microcat_id` используется через статистику подкатегорий.

Текстовые индексы строятся по исходному корпусу объявлений.


In [3]:
indexes = {}
fields = {"title": corpus.item_title_raw,
          "description": corpus.item_description_raw.str.slice(0, 4000),
          "params": corpus.item_infm_params_text}
for name, texts in fields.items():
    indexes[name] = BM25(b=0.5 if name == "title" else 0.75).fit([stem_text(x) for x in texts])
    print("Построен индекс:", name)
indexes["char"] = CharacterIndex().fit(corpus.item_title_raw.map(normalize))

item_vectors = np.load("artifacts/item_embeddings.npy", mmap_mode="r")
query_vectors = np.load("artifacts/query_embeddings.npy", mmap_mode="r")
vector_ids = pd.read_parquet("artifacts/embedding_query_ids.parquet").query_id
query_vectors = dict(zip(vector_ids, query_vectors))
assert item_vectors.shape == (len(corpus), 384)


Построен индекс: title


Построен индекс: description


Построен индекс: params


### Расчёт эмбеддингов

Использована модель [intfloat/multilingual-e5-small](https://huggingface.co/intfloat/multilingual-e5-small), revision `614241f622f53c4eeff9890bdc4f31cfecc418b3`, без дообучения. Представления получены усреднением токенов без padding и L2-нормализацией.

- Объявление: префикс `passage:`, заголовок, первые 1 400 символов описания и 300 символов параметров; максимум 160 токенов.
- Запрос: префикс `query:` и текст запроса; максимум 64 токена.

По умолчанию загружаются сохранённые эмбеддинги. Ниже приведена функция их расчёта. Для пересчёта нужны локальные веса, PyTorch 2.14.0 и Transformers 5.17.0. Исходный расчёт выполнен на MPS в float16; пересчёт на другом устройстве может изменить ближайших соседей с почти равными оценками.


In [4]:
def encode_e5(texts, model_dir, device="cpu", max_length=160):
    import torch
    from transformers import AutoModel, AutoTokenizer
    torch.manual_seed(42)
    torch.set_num_threads(4)
    tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    model = AutoModel.from_pretrained(model_dir, local_files_only=True).to(device).eval()
    if device in ("mps", "cuda"):
        model = model.half()
    vectors = []
    with torch.inference_mode():
        for start in range(0, len(texts), 64):
            batch = tokenizer(texts[start:start+64], padding=True, truncation=True,
                              max_length=max_length, return_tensors="pt")
            batch = {name: value.to(device) for name, value in batch.items()}
            hidden = model(**batch).last_hidden_state.float()
            mask = batch["attention_mask"].unsqueeze(-1).bool()
            pooled = hidden.masked_fill(~mask, 0).sum(1) / mask.sum(1).clamp(min=1)
            vectors.append(torch.nn.functional.normalize(pooled, p=2, dim=1).cpu().numpy())
    return np.concatenate(vectors)

# Тексты для расчёта эмбеддингов объявлений и запросов:
# passages = ["passage: " + normalize(t) + ". " + normalize(d)[:1400] + ". " + normalize(p)[:300]
#             for t, d, p in corpus[["item_title_raw", "item_description_raw", "item_infm_params_text"]].itertuples(index=False, name=None)]
# query_texts = ["query: " + normalize(t) for t in pd.concat([evaluation, queries]).search_query]


## 3. Эксперименты и обучение отбора

Результаты на dev-выборке. Базовый BM25 складывает оценки заголовка и описания с весами 1 и 0.4:

| Вариант | Recall@50 |
|---|---:|
| BM25 по заголовку и описанию | 0.2662 |
| Текстовый поиск с расстоянием | 0.8152 |
| Текстовый поиск + история локаций + CatBoost | 0.9061 |
| Текстовый и семантический поиск + история локаций + CatBoost | **0.9565** |

Полнота расширенного гибридного пула — 0.9802. На 968 dev-запросах, у которых все выбранные объявления отсутствуют в истории взаимодействий, Recall@50 — 0.9498.

Из 100, 200, 300, 400 и 500 деревьев по dev выбраны **400**. Затем один раз проверена отдельная выборка из 2 000 запросов: **Recall@50 = 0.9409**, полнота пула — 0.9736. После этого модель не менялась. Метрики рассчитаны на отложенных данных из train; разметка бенчмарка недоступна.

При `RUN_EXPERIMENTS = True` выполняются обучение и оценка гибрида, включая текстовые базовые методы. Итоговый CSV в обоих режимах формируется с сохранённой моделью из архива.


In [5]:
def collect_pairs(engine, frame, for_training=False):
    rows, labels, pointers, denominators = [], [], [0], []
    baseline = {}
    for number, query in enumerate(frame.itertuples(index=False)):
        candidates, features, methods = engine.retrieve(query)
        relevant = set(query.relevant)
        target = np.isin(engine.ids[candidates], list(relevant)).astype(np.int8)
        # Обучающие пары формируются только из найденных кандидатов.
        # Отрицательные примеры выбираются из того же пула.
        if for_training and len(candidates) > 300:
            positive, negative = np.flatnonzero(target), np.flatnonzero(target == 0)
            rng = np.random.default_rng(42 + number)
            keep = np.sort(np.concatenate([positive, rng.choice(
                negative, min(len(negative), max(0, 300-len(positive))), replace=False)]))
        else:
            keep = np.arange(len(candidates))
        rows.append(features[keep]); labels.append(target[keep])
        pointers.append(pointers[-1] + len(keep)); denominators.append(len(relevant))
        for name, ids in methods.items():
            baseline.setdefault(name, []).append(len(set(engine.ids[ids]) & relevant) / len(relevant))
    return np.concatenate(rows), np.concatenate(labels), pointers, denominators, baseline

def recall50(scores, dataset):
    _, labels, pointers, denominators, _ = dataset
    return np.mean([labels[start:end][topk(scores[start:end], 50)].sum() / total
                    for start, end, total in zip(pointers[:-1], pointers[1:], denominators)])

if RUN_EXPERIMENTS:
    experiment_engine = Retriever(corpus, train[train.text_bucket < 70], indexes, item_vectors, query_vectors)
    fit = collect_pairs(experiment_engine, evaluation[evaluation.split == "fit"], for_training=True)
    dev = collect_pairs(experiment_engine, evaluation[evaluation.split == "dev"])
    candidate_model = CatBoostClassifier(iterations=500, depth=7, learning_rate=0.06,
        loss_function="Logloss", l2_leaf_reg=5, random_seed=42, thread_count=4,
        class_weights=[1, 25], allow_writing_files=False, verbose=100)
    candidate_model.fit(Pool(fit[0], fit[1], feature_names=FEATURES))
    checkpoints = {n: recall50(candidate_model.predict_proba(dev[0], ntree_end=n)[:, 1], dev)
                   for n in range(100, 501, 100)}
    best = max(checkpoints, key=checkpoints.get)
    candidate_model.shrink(best)
    print("Dev:", checkpoints, "выбрано деревьев:", best)
    print("Базовые методы:", {name: float(np.mean(values)) for name, values in dev[4].items()})
    test = collect_pairs(experiment_engine, evaluation[evaluation.split == "test"])
    print("Финальная проверка:", recall50(candidate_model.predict_proba(test[0])[:, 1], test))
    del experiment_engine, fit, dev, test, candidate_model


### Ошибки и ограничения

1. **Общий заголовок скрывает услугу.** «Увеличение губ» → «Косметолог», «прочистка канализации» → «Сантехник». Поиск по описанию и E5 добавили кандидатов без буквального совпадения заголовков.
2. **Исполнитель находится в другом городе.** Оставлены глобальные кандидаты и добавлена статистика переходов между локациями вместо жёсткого фильтра.
3. **У нового объявления нет истории.** Нулевая популярность не исключает его; работают текст, векторы и характеристики объявления. Этот срез проверен отдельно.
4. **Переобучение отбора.** Увеличение числа деревьев ухудшало качество на dev. Число деревьев выбрано по dev Recall@50, а не по ошибке обучения.

Не все подходящие объявления отмечены как выбранные: отрицательные примеры могут быть семантически релевантны. Запросы бенчмарка в среднем длиннее запросов dev (3.20 против 2.34 слова) и реже содержат фильтры (36.9% против 67.4%). Большинство объявлений бенчмарка не встречается в train, поэтому качество на объявлениях без истории проверено отдельно.


## 4. Ответ для бенчмарка

Для итогового поиска статистика взаимодействий рассчитывается по всему train. Кандидаты оцениваются сохранённой моделью CatBoost. В `answer.csv` записываются `query_id` и до 50 идентификаторов объявлений из `benchmark_items`.


In [6]:
engine = Retriever(corpus, train, indexes, item_vectors, query_vectors)
model = CatBoostClassifier()
model.load_model("artifacts/selector_dense.cbm")
allowed = corpus.item_id.isin(bench.item_id).to_numpy()
answers = []
for number, query in enumerate(queries.itertuples(index=False)):
    candidates, features, _ = engine.retrieve(query, allowed)
    scores = model.predict_proba(features)[:, 1]
    selected = candidates[topk(scores, 50)]
    answers.append(" ".join(engine.ids[selected]))
    if (number + 1) % 500 == 0:
        print(f"Готово {number + 1}/{len(queries)} запросов")
answer = pd.DataFrame({"query_id": queries.query_id, "answer": answers})
answer.to_csv("answer.csv", index=False)


Готово 500/2452 запросов


Готово 1000/2452 запросов


Готово 1500/2452 запросов


Готово 2000/2452 запросов


## 5. Проверка файла

Проверяются колонки, полнота списка запросов, формат идентификаторов, отсутствие повторов и ограничение в 50 объявлений. SHA-256 сравнивается с контрольной суммой отправленного `answer.csv`.


In [7]:
saved = pd.read_csv("answer.csv", dtype=str, keep_default_na=False)
assert list(saved.columns) == ["query_id", "answer"]
assert len(saved) == len(queries) == 2452
assert saved.query_id.is_unique and set(saved.query_id) == set(queries.query_id)
assert saved.query_id.str.len().eq(16).all()
valid_ids = set(bench.item_id)
for value in saved.answer:
    ids = value.split(" ")
    assert len(ids) == len(set(ids)) == 50
    assert all(re.fullmatch(r"[0-9a-f]{16}", item) for item in ids)
    assert set(ids) <= valid_ids
assert sha256("answer.csv") == EXPECTED_ANSWER, "CSV отличается от отправленного результата"
print("Проверено: 2452 запроса, по 50 уникальных item_id. CSV совпал байт в байт.")
print("SHA-256:", sha256("answer.csv"))


Проверено: 2452 запроса, по 50 уникальных item_id. CSV совпал байт в байт.
SHA-256: 04a5deafb9f6037ac2c32a069cff1a771f4a22d69b234b5640ff66ca377e0b0b
